# Iterations: rust vs. DSim (iterations-vs-dsim-2026)

Data: `~/runs-svn/matsim-rust/iterations-vs-dsim-2026` (experiments `scaling-v2` (rust) and `dsim-scaling` (DSim)), fetched with
`uv run scripts/cluster/cluster.py fetch <experiment>`. Loading and figures are in `src/main/python/iterations/`, so this
notebook and `cd src/main/python && uv run --group analysis python -m iterations` (all PDFs at once) use the same definitions.

Start Jupyter with `uv run --group analysis jupyter lab`.

**Definitions**
- *Real-time ratio* = simulated seconds (36 h) / wall-clock seconds of the simulation loop of one iteration. rust: span
  `simulation::run`, DSim: the ratio it logs itself (same loop).
- *Phases* per iteration: mobsim (rust: the whole mobsim phase including its preparation, DSim: the simulation loop),
  scoring, replanning (before the mobsim of the iteration). Plot 2 leaves out the rest.
- Iteration 0 includes the setup and has no replanning. Real-time ratio: median over iterations 1-9 (band: min-max),
  phases: median over iterations 1-8.
- 0 %: empty population on the same network and partitions, i.e. the synchronization alone.

In [ ]:
%config InlineBackend.figure_format = 'retina'
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from iterations.load import load_phases, load_runs, phase_summary, rtr_summary
from iterations.plots import phase_lines, rtr_no_agents, rtr_vs_parts, save, scoring_overhead

In [ ]:
runs = load_runs()
phases = load_phases(runs)
rtr = rtr_summary(phases)
shares = phase_summary(phases)
runs.groupby(["engine", "pct", "scoring", "events"]).size().unstack("pct", fill_value=0)

## Real-time ratio per run

In [ ]:
rtr[rtr["scoring"]].pivot_table(index="parts", columns=["engine", "pct"], values="rtr").round(0)

## Plot 1a: real-time ratio without agents (0 %)

Caption: *Real-time ratio (RTR) of the mobsim over the number of partitions without agents (empty population on the same
network and partitions), i.e. the synchronization alone, for the rust implementation and DSim. Median over iterations
1-9, the band shows minimum and maximum. Maxima and the values at 192 partitions are labeled.*

In [ ]:
fig = rtr_no_agents(rtr)
paths = save(fig, "rtr_no_agents")
display(fig)
plt.close(fig)
paths

## Plot 1b: real-time ratio over the number of partitions

Caption: *Real-time ratio (RTR) of the mobsim over the number of partitions for the rust implementation and DSim, one
panel per population share. Median over iterations 1-9, the band shows minimum and maximum. The maximum of each line is
marked. Both with scoring.*

In [ ]:
fig = rtr_vs_parts(rtr)
paths = save(fig, "rtr_vs_parts")
display(fig)
plt.close(fig)
paths

## Plot 2: phases of an iteration

Caption: *Wall time per iteration of mobsim, scoring and replanning over the number of partitions, one panel per
population share, rust solid and DSim dashed. Median over iterations 1-8, values below 1 ms are left out. The mobsim of
rust includes its preparation, that of DSim is its simulation loop.*

In [ ]:
fig = phase_lines(shares)
paths = save(fig, "phase_lines")
display(fig)
plt.close(fig)
paths

In [ ]:
# the numbers behind plot 2: median seconds per phase
shares.pivot_table(index=["engine", "pct", "parts"], columns="phase", values="seconds").round(2)

## Plot 3: scoring on/off (rust)

Caption: *Left: real-time ratio (RTR) of the rust mobsim with and without scoring. Right: additional wall time of the
mobsim with scoring. Median over iterations 1-9.*

In [ ]:
fig = scoring_overhead(rtr)
paths = save(fig, "scoring_overhead")
display(fig)
plt.close(fig)
paths